# 07. Spatial Extremes

This notebook uses `RMC.BestFit.Model.SpatialExtremes.GaussianSpatialGEV` directly and compares correlation-function assumptions via BestFit log-likelihood.

In [4]:
import pythonnet
pythonnet.load("netfx")

import clr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helper_functions import resolve_bestfit_dll, resolve_numerics_dll, convert_to_donet_array

clr.AddReference(str(resolve_numerics_dll()))
clr.AddReference(str(resolve_bestfit_dll()))

from System import Array, Double
from RMC.BestFit.Model import GeneralLinearFunction
from RMC.BestFit.Model.SpatialExtremes import GaussianSpatialGEV, CorrelationFunctionType

def to_dotnet_2d(a):
    a = np.asarray(a, dtype=float)
    arr = Array.CreateInstance(Double, a.shape[0], a.shape[1])
    for i in range(a.shape[0]):
        for j in range(a.shape[1]):
            arr[i, j] = float(a[i, j])
    return arr

In [5]:
rng = np.random.default_rng(25)
coords = np.array([[0.0,0.0],[1.0,0.0],[0.0,1.0],[1.0,1.0],[2.0,0.5]])
index_flood = np.array([8000, 9000, 11000, 10000, 12000], dtype=float)
n_years = 30

# At-site annual maxima matrix: rows=years, cols=sites
at_site = np.vstack([index_flood * rng.lognormal(0, 0.18, size=len(index_flood)) for _ in range(n_years)])

# Intercept-only linear functions for location/scale/shape components
X = np.ones((coords.shape[0], 1), dtype=float)
loc_fun = GeneralLinearFunction('Location', to_dotnet_2d(X))
sca_fun = GeneralLinearFunction('Scale', to_dotnet_2d(X))
shp_fun = GeneralLinearFunction('Shape', to_dotnet_2d(X))

corr_types = [
    CorrelationFunctionType.Exponential,
    CorrelationFunctionType.PoweredExponential,
    CorrelationFunctionType.Spherical,
]

rows = []
models = {}
for ctype in corr_types:
    m = GaussianSpatialGEV(to_dotnet_2d(at_site), to_dotnet_2d(coords), ctype, loc_fun, sca_fun, shp_fun)
    m.SetDefaultParameters()
    p0 = np.array([float(p.Value) for p in m.Parameters], dtype=float)
    ll = float(m.LogLikelihood(convert_to_donet_array(p0)))
    rows.append({'correlation_model': str(ctype), 'log_likelihood_at_default': ll, 'n_parameters': int(m.NumberOfParameters)})
    models[str(ctype)] = (m, p0)

summary = pd.DataFrame(rows).sort_values('log_likelihood_at_default', ascending=False)
summary

NullReferenceException: Object reference not set to an instance of an object.
   at RMC.BestFit.Model.SpatialExtremes.GaussianSpatialGEV.SetDefaultParameters()
   at RMC.BestFit.Model.SpatialExtremes.GaussianSpatialGEV..ctor(Double[,] atSiteData, Double[,] coordinates, CorrelationFunctionType correlationType, GeneralLinearFunction location, GeneralLinearFunction scale, GeneralLinearFunction shape)

In [ ]:
# Derive quick site-level Q100 values from the top-ranked model's current parameter state
best_name = summary.iloc[0]['correlation_model']
best_model, best_params = models[best_name]
best_model.SetParameterValues(convert_to_donet_array(best_params))

# Read current linear-function intercepts from model component parameters
loc0 = float(best_model.Location.Parameters[0].Value)
sca0 = float(best_model.Scale.Parameters[0].Value)
shp0 = float(best_model.Shape.Parameters[0].Value)

# Approximate at-site Q100 from the fitted marginal component values (same marginal form here)
from Numerics.Distributions import GeneralizedExtremeValue
gev = GeneralizedExtremeValue(loc0, max(1e-6, abs(sca0)), shp0)
q100 = float(gev.InverseCDF(0.99))

print('Best correlation model by default-parameter log-likelihood:', best_name)
print('Marginal parameter snapshot (location, scale, shape):', (round(loc0,3), round(sca0,3), round(shp0,3)))
print('Representative marginal Q100:', round(q100, 2))

In [ ]:
plt.figure(figsize=(8,5))
for s in range(at_site.shape[1]):
    plt.scatter(np.repeat(s+1, at_site.shape[0]), at_site[:, s], s=12, alpha=0.35)
plt.xticks(range(1, at_site.shape[1]+1))
plt.xlabel('Site index')
plt.ylabel('Annual peak')
plt.title('Spatial Extremes Input Data (Site-wise)')
plt.grid(alpha=0.25)
plt.tight_layout()

## Summary and Exercises

You used BestFit `GaussianSpatialGEV` directly and compared spatial correlation assumptions with BestFit likelihood values.

Recommended exercises:
1. Add covariates (elevation, area) to `GeneralLinearFunction` instead of intercept-only.
2. Expand to more sites and test sensitivity to coordinate scaling.
3. Build Q100 maps using site-specific marginals from fitted component functions.